# KoBERT · 한국어 10종 다중 라벨

원본 `finetuned_bert (2).ipynb`의 0-based cell 0를 변경 없이 추출했습니다.

원본 SHA-256: `e052814403adca3379fa0fd8c4731970940a415488595e3c1b5db277cc29006c`

원본 실행 출력·위젯 메타데이터·원시 데이터는 포함하지 않습니다. 저장된 과거 결과와 입력 준비 방법은 README를 참고하세요. 이 공개 사본은 재학습하지 않았으며, 실행에는 별도 CSV가 필요합니다. v4 라벨 매핑과 분할 중복 검토 후 재평가가 필요합니다.


In [ ]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer, AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import label_ranking_average_precision_score, f1_score, roc_auc_score
from datasets import load_dataset
import torch
import numpy as np
import re

from dataclasses import dataclass

# KoBert Model
model_from = 'monologg/kobert'
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_from)

# 정규표현식 기반 일부 변형 표현 치환 적용된 데이터셋 활용
data_files = {
    "train": "train_augmented.csv",
    "valid": "valid_augmented.csv"
}
dataset = load_dataset("csv", data_files=data_files, delimiter=",")

tokenizer = AutoTokenizer.from_pretrained(model_from)

# 기존 메소드 백업
_orig_save_vocabulary = tokenizer.save_vocabulary

def save_vocabulary_compat(save_directory, filename_prefix=None):
    # filename_prefix는 무시하고 기존 방식대로 저장
    return _orig_save_vocabulary(save_directory)

# monkey patch
tokenizer.save_vocabulary = save_vocabulary_compat

def parse_labels_str(s):
    """
    CSV에서 읽어온 labels가 문자열일 때
    ex. "[0 0 0 0 0 0 0 0 0 1]" 또는 "[0, 0, ...]"
    -> [0,0,...] 리스트[int]로 변환
    """
    if not isinstance(s, str):
        return s

    s = s.strip()
    s = s.replace("[", "").replace("]", "")
    parts = re.split(r"[,\s]+", s)
    parts = [int(p) for p in parts if p != ""]
    return parts

@dataclass
class DataCollatorWithFloatLabels(DataCollatorWithPadding):
    def __call__(self, features):
        # 1. labels 만 따로 꺼내기
        labels = [f["labels"] for f in features]

        # 2. 원래 features 에서는 labels 를 제거 (부모 collator 는 labels 모르는 상태여야 함)
        for f in features:
            f.pop("labels")

        # 3. input_ids / attention_mask / token_type_ids 등은 부모가 처리
        batch = super().__call__(features)

        # 4. labels를 float32 tensor 로 변환 (각 sample 별로)
        #    - list 든 tensor 든 상관없이 as_tensor 가 알아서 받아줌
        label_tensors = [torch.as_tensor(l, dtype=torch.float) for l in labels]

        # 5) (batch_size, num_labels) 형태로 쌓기
        batch["labels"] = torch.stack(label_tensors)

        return batch

collator = DataCollatorWithFloatLabels(tokenizer=tokenizer)

# Dataset labels
labels = [
    "여성/가족",
    "남성",
    "성소수자",
    "인종/국적",
    "연령",
    "지역",
    "종교",
    "기타 혐오",
    "악플/욕설",
    "clean"
]

# Batch size
batch_size = 64

def preprocess(examples):
    tokenized = tokenizer(
        examples["문장"],
        truncation=True,
        padding=False,
        max_length=128,
    )

    raw_labels = examples["labels"]
    processed_labels = [parse_labels_str(s) for s in raw_labels]
    tokenized["labels"] = processed_labels
    return tokenized

# def compute_metrics(x):
#   return {'lrap': label_ranking_average_precision_score(x.label_ids, x.predictions)}

def compute_metrics(x):
    # x.predictions: [batch, num_labels] logits
    # x.label_ids:   [batch, num_labels] 0/1 라벨

    logits = x.predictions
    labels = x.label_ids

    # sigmoid 계산
    probs = 1 / (1 + np.exp(-logits))

    lrap = label_ranking_average_precision_score(labels, probs)

    # 3) F1 score 계산, threshold 는 0.5
    y_pred = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(labels, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, y_pred, average="macro", zero_division=0)

    return {
        "lrap": lrap,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }

tokenized_dataset = dataset.map(preprocess, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'labels', 'attention_mask', 'token_type_ids'])

num_labels = len(labels)

# Sequence classification task 정의된 model
model = BertForSequenceClassification.from_pretrained(
    model_from,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

model.config.id2label = {i: label for i, label in zip(range(num_labels), labels)}
model.config.label2id = {label: i for i, label in zip(range(num_labels), labels)}

# Training args
train_args = TrainingArguments(
    output_dir="gentletalk_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=8,
    load_best_model_at_end=True,
    metric_for_best_model="lrap",
    greater_is_better=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=collator
)

# Train
trainer.train()


# Best model 저장
best_dir = trainer.state.best_model_checkpoint
model = BertForSequenceClassification.from_pretrained(best_dir)
model.save_pretrained("gentletalk_model")